# Code Book — Week 5, Anomaly Detection II

**Programme:** Applied Machine Learning for Operations — Model Builder Track
**Week:** 5 — Anomaly Detection II, Machine Learning
**AI-use mode:** AI-Augmented. The assistant may write code; you verify and explain every line.
**Files:** 2 CSVs in `data/`

---

## What this week needs from the data

Week 4 detected faults one channel at a time. Week 5 exists to find the faults that method structurally cannot reach — where the **relationship between channels** breaks while every individual channel stays comfortably inside its normal range.

That demands a dataset with three properties the Week 4 files did not need:

1. **Several channels that genuinely move together.** Array current drives state of charge; RF drive level drives amplifier temperature. Those couplings have to be real, or a multivariate model has nothing to learn.
2. **Faults that are univariately invisible.** Three of the five labelled events here never take any single channel outside its own 3-sigma band. They are the entire case for the week.
3. **Enough history for a temporal split.** A full year of hourly data, so you can train on eight months and test on four without running out of rows.

## Provenance

This is a **synthetic teaching dataset**, not published NIGCOMSAT telemetry. It was specified to reproduce the real behaviour of a GEO satellite at 42.5°E — eclipse seasons around the equinoxes, eclipse at roughly satellite local midnight, battery discharge depth tracking eclipse duration, amplifier temperature tracking RF drive — so the methods land on data that behaves like the real thing.

Every figure is synthetic, so this file may be used freely in AI assistants. Real NIGCOMSAT telemetry remains subject to the Week 2 data restrictions.

**Instructors:** `anomaly_labels_week5.csv` is the answer key. Withhold it until participants have run their own detectors — topic 2's exercise depends on them discovering the multivariate events for themselves.

## The two files

| File | Rows | Span | Purpose |
|---|---|---|---|
| `bus_telemetry_multivariate_hourly.csv` | 8,760 | 2025-01-01 to 2025-12-31, hourly | Six coupled channels, five embedded faults |
| `anomaly_labels_week5.csv` | 5 | — | Ground truth, with a flag for univariate detectability |

### Which topic uses what

| Topic | Uses | For |
|---|---|---|
| 5.1 The scikit-learn workflow | the 6 raw channels | Building X, scaling, pipelines |
| 5.2 Isolation forest and clustering | the 6 raw channels + labels | First multivariate detector |
| 5.3 Feature engineering | all channels | Lags, rolling stats, cross-channel ratios |
| 5.4 Temporal train/test discipline | engineered features + labels | Honest evaluation against the Week 4 baseline |

---

## `bus_telemetry_multivariate_hourly.csv`

A full year of hourly electrical-power and RF telemetry from a single GEO platform. Six numeric channels plus one state flag.

**Shape:** 8,760 rows — 2025-01-01T00:00 to 2025-12-31T23:00, one row per hour, no gaps, no duplicates.

### Columns

| Column | Type | Unit | Description |
|---|---|---|---|
| `timestamp` | datetime (hourly) | — | Hour of the reading, UTC |
| `solar_array_current_a` | float | A | Total array output. About 27.8 A in sunlight, near 0 in eclipse, elevated during post-eclipse recharge |
| `bus_voltage_v` | float | V | Regulated bus voltage. About 50.4 V nominal, sagging to roughly 46 V on battery |
| `battery_soc_pct` | float | % | State of charge, 0–100. Falls to about 77% at the deepest eclipse |
| `battery_temp_c` | float | °C | Pack temperature. About 18.2 °C nominal, rising on discharge and recharge |
| `twta_temp_c` | float | °C | Amplifier baseplate temperature, roughly 59–67 |
| `rf_drive_pct` | float | % | RF drive as a share of saturation, rising from about 70% to 81% across the year |
| `eclipse_flag` | int | 0/1 | 1 when the satellite is in eclipse that hour |

### The physical couplings the model must learn

These are the relationships that make the dataset multivariate rather than six unrelated series:

- **Array current → state of charge.** In sunlight the battery charges and sits at 100%. During eclipse the array drops to zero and SoC falls in proportion to eclipse duration. After eclipse, array current rises above nominal while the battery recharges.
- **Array current → bus voltage.** On battery the bus sags by about 3.5 V. The two move together.
- **Eclipse → battery temperature.** Discharge and recharge both warm the pack by a few degrees.
- **RF drive → amplifier temperature.** About 0.085 °C per percentage point of drive. This is the coupling that topic 3's `temp_per_drive` feature exploits.

### Seasonal structure

- **Two eclipse seasons**, centred near 21 March and 23 September, each about 45 days wide, peaking at 72 minutes a day. 92 eclipse hours in the year.
- Eclipse occurs at **21:00 UTC**, close to satellite local midnight at 42.5°E. The natural period is 24, not 7.
- A mild annual cycle on array current and amplifier temperature from sun angle.
- `rf_drive_pct` carries a genuine upward trend as capacity is taken up — which is why topic 3 warns that amplifier temperature and drive level correlate mostly through shared trend.

## Setup

Run this first. It assumes the notebook sits next to the `data/` folder.

**What this cell does.** Imports the libraries, loads both CSVs with the timestamp parsed and set as the index, and declares hourly frequency so a missing hour would appear as `NaN` rather than vanish. Run it first — everything below depends on `bus` and `labels`.

**What to read in the output.** 8,760 rows, zero gaps, 92 eclipse hours. In `describe()`, note how different the six ranges are: state of charge spans 77–100 while array current spans 0–29.8. That difference is why topic 1 insists on scaling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path("data")
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)

bus = pd.read_csv(DATA / "bus_telemetry_multivariate_hourly.csv",
                  parse_dates=["timestamp"], index_col="timestamp").asfreq("H")

labels = pd.read_csv(DATA / "anomaly_labels_week5.csv",
                     parse_dates=["start", "end"])

print("rows:", len(bus), " gaps:", int(bus.isna().sum().sum()))
print("eclipse hours:", int(bus["eclipse_flag"].sum()))
bus.describe().round(2)

### Confirming the couplings are real

Before trusting any multivariate model, check that the channels actually move together. If they do not, a multivariate detector has nothing to add over six separate control charts.

**What to read in the output.** The correlation matrix confirms the couplings are real — array current, bus voltage and state of charge move together, and amplifier temperature tracks drive level. If these were near zero, a multivariate model would have nothing to add over six separate control charts.

The second table shows the two operating states side by side: in eclipse, array current collapses to near zero, the bus sags about 3.5 V, and state of charge drops. That is the platform behaving exactly as designed, and a detector must not treat it as a fault.

In [ ]:
sunlit = bus[bus["eclipse_flag"] == 0]

print("correlation, sunlit hours only:")
print(sunlit[["solar_array_current_a", "bus_voltage_v", "battery_soc_pct",
              "twta_temp_c", "rf_drive_pct"]].corr().round(2))

print("\nmean by operating state:")
print(bus.groupby("eclipse_flag")[["solar_array_current_a", "bus_voltage_v",
                                   "battery_soc_pct", "battery_temp_c"]].mean().round(2))

---

## `anomaly_labels_week5.csv` — the answer key

Five labelled events. The `detectable_univariately` column is the one that matters for this week: it records whether any single-channel method from Week 4 could have found the event.

### Columns

| Column | Type | Description |
|---|---|---|
| `event_id` | string | EV01 to EV05 |
| `start` | datetime | First affected hour |
| `end` | datetime | Last affected hour. Equals `start` for single-hour events |
| `type` | string | `point`, `correlation_break` or `thermal_decoupling` |
| `channels_involved` | string | Semicolon-separated channel names |
| `detectable_univariately` | string | `yes` or `no` — whether a Week 4 method could find it |
| `operational_cause` | string | The plain-language explanation an engineer would log |

### The five events

| ID | When | Type | Univariate? | What happened |
|---|---|---|---|---|
| EV01 | 19 Feb, 1 hour | point | **yes** | Partial array string dropout. Array current falls to 58% of nominal, battery temperature 4.4 °C high |
| EV02 | 1 May, 12 hours | correlation break | **no** | Charge controller fault. Array current normal at about 27.0 A, but SoC sits near 93% instead of recovering to 100% |
| EV03 | 29 Jun, 1 hour | point | **yes** | Partial array string dropout, same signature as EV01 |
| EV04 | 3 Sep – 30 Sep, 672 hours | thermal decoupling | **no** | Cooling path degradation. Amplifier runs about 2.9 °C hot for its drive level. Peak raw temperature 67.0 °C against a 3-sigma limit of 68.2 — never breaches |
| EV05 | 17 Nov, 24 hours | correlation break | **no** | Charge controller fault recurrence |

### Why three of these defeat Week 4 entirely

**EV02 and EV05.** Array current during the fault runs 26.93 to 27.14 A. The sunlit 3-sigma band is 26.49 to 29.18 A — every reading is comfortably inside. State of charge sits around 93%, well above any sensible floor and higher than a normal eclipse night. Neither channel alone is remarkable. The *pair* — full sun with the battery refusing to recover — has never occurred before.

**EV04.** Raw amplifier temperature peaks at 67.0 °C against a 3-sigma ceiling of 68.2 °C, so no control chart fires. But modelled against drive level and sun angle, the fault period sits about **22 sigma** from the fitted relationship. One derived feature converts the least detectable event in the file into the most obvious.

**What to read in the output.** All five events in one table. Look at the `detectable_univariately` column: only EV01 and EV03 are `yes`. The other three are the reason this week exists. Note also the spans — EV04 covers 672 hours, which forces the event-level scoring convention rather than per-row.

In [ ]:
labels

### Proving EV02 is univariately invisible

Run this before topic 2. It is the empirical case for the whole week.

**What to read in the output.** This is the empirical case for the week. The sunlit 3-sigma band runs roughly 26.5 to 29.2 A. Every EV02 array reading sits between 26.93 and 27.14 — comfortably inside, so the check returns `False`. Meanwhile state of charge sits near 93% instead of the usual 99.7%. Neither channel alone is remarkable; the combination has never occurred.

In [ ]:
arr = sunlit["solar_array_current_a"]
m, s = arr.mean(), arr.std()
print(f"sunlit array current: mean={m:.2f} A  sd={s:.3f} A")
print(f"3-sigma band        : {m - 3*s:.2f} to {m + 3*s:.2f} A")

ev02 = bus.loc["2025-05-01 01:00":"2025-05-01 12:00"]
ev02_sunlit = ev02[ev02["eclipse_flag"] == 0]["solar_array_current_a"]
print("\nEV02 array current values:")
print(ev02_sunlit.round(2).values)
print("any outside the 3-sigma band?",
      bool(((ev02_sunlit - m).abs() > 3 * s).any()))

soc = bus.loc["2025-05-01 01:00":"2025-05-01 12:00", "battery_soc_pct"]
print(f"\nEV02 state of charge: {soc.min():.1f} to {soc.max():.1f} %")
print("normal sunlit SoC   :", round(sunlit['battery_soc_pct'].mean(), 1), "%")

### Proving EV04 needs a derived feature

Fit amplifier temperature against drive level and sun angle on the clean first five months, then measure the fault period against that fit.

**What to read in the output.** Amplifier temperature is fitted against drive level and sun angle on the clean first five months, then the September fault period is measured against that fit. Expect a baseline residual standard deviation of about 0.18 °C and an EV04 mean residual near 3.9 °C — roughly **22 sigma**.

Then the contrast: the univariate check returns `False`, because the peak raw temperature of 67.0 °C never reaches the 3-sigma ceiling of 68.2 °C. Same data, same event; one derived feature turns the least detectable fault in the file into the most obvious.

In [ ]:
doy = bus.index.dayofyear
design = pd.DataFrame({
    "drive": bus["rf_drive_pct"],
    "sin":   np.sin(2 * np.pi * doy / 365),
    "cos":   np.cos(2 * np.pi * doy / 365),
}, index=bus.index)
design["const"] = 1.0

clean = design.loc[:"2025-05-31"]
coef, *_ = np.linalg.lstsq(clean.values,
                           bus.loc[:"2025-05-31", "twta_temp_c"].values,
                           rcond=None)

expected = design.values @ coef
resid = pd.Series(bus["twta_temp_c"].values - expected, index=bus.index)

base = resid.loc[:"2025-05-31"]
ev04 = resid.loc["2025-09-03":"2025-09-30"]

print(f"baseline residual sd : {base.std():.3f} C")
print(f"EV04 mean residual   : {ev04.mean():.2f} C")
print(f"                     = {(ev04.mean() - base.mean()) / base.std():.1f} sigma")

tw = bus["twta_temp_c"]
print(f"\nunivariate check: 3-sigma ceiling {tw.mean() + 3*tw.std():.2f} C")
print(f"EV04 peak raw value          : {bus.loc['2025-09-03':'2025-09-30', 'twta_temp_c'].max():.2f} C")
print("univariately flagged?", bool(bus.loc["2025-09-03":"2025-09-30", "twta_temp_c"].max() > tw.mean() + 3*tw.std()))

---

## The feature matrix

Topic 3's builder. Every window is **trailing** and every shift is **positive** — nothing here uses information that would not exist at the moment the reading arrives.

The test to apply to any feature you add: *could I compute this at 14:00 using only what I knew at 14:00?*

**What this cell does.** Builds the feature matrix for topics 3 and 4. For four channels it creates the raw value, two lags, a first difference, a 24-hour rolling mean and standard deviation, and the deviation from that mean. Then two cross-channel ratios and two known-physics columns.

**What to read in the output.** 32 columns from 6 raw channels, and 24 rows lost to the longest window. Every window is trailing and every shift positive — nothing here uses information that would not exist at the moment the reading arrived.

In [ ]:
def make_features(df):
    """Trailing-only features. No centred windows, no negative shifts."""
    f = pd.DataFrame(index=df.index)

    for c in ["solar_array_current_a", "battery_soc_pct",
              "twta_temp_c", "bus_voltage_v"]:
        f[c]              = df[c]
        f[f"{c}_lag1"]    = df[c].shift(1)
        f[f"{c}_lag24"]   = df[c].shift(24)
        f[f"{c}_d1"]      = df[c].diff(1)
        f[f"{c}_mean24"]  = df[c].rolling(24).mean()
        f[f"{c}_std24"]   = df[c].rolling(24).std()
        f[f"{c}_dev24"]   = df[c] - f[f"{c}_mean24"]

    # cross-channel: the couplings, as ratios
    f["temp_per_drive"] = df["twta_temp_c"] / df["rf_drive_pct"]
    f["soc_gap"]        = 100 - df["battery_soc_pct"]

    # known physics, handed over rather than rediscovered
    f["eclipse_flag"] = df["eclipse_flag"]
    f["hour"]         = df.index.hour

    return f.dropna()


F = make_features(bus)
print("feature matrix:", F.shape)
print("rows lost to the longest window:", len(bus) - len(F))
list(F.columns)

### Checking for near-duplicate features

Isolation forest picks its split dimension at random, so a column that duplicates another effectively double-weights that channel. Twenty to forty columns is a sensible target from six raw channels.

**What to read in the output.** The most correlated feature pairs. Anything above about 0.98 is effectively a duplicate column, and duplicates double-weight that channel when isolation forest picks split dimensions at random. Use this to prune before fitting, not after.

In [ ]:
c = F.corr().abs()
pairs = (c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
          .stack().sort_values(ascending=False))
print("most redundant feature pairs:")
print(pairs.head(10).round(3))

---

## The temporal split

The rule: every training timestamp must precede every test timestamp, and an **embargo** at the boundary must be at least as long as the longest feature window — otherwise a 24-hour rolling mean on the first test row reaches back into training data.

A random split on this data will report a better score than the temporal one, on identical features and an identical model. That improvement is entirely fictional.

**What to read in the output.** The split boundaries and the row counts. The `assert` is the important line — it fails loudly if training ever overlaps test. Expect about 48 embargoed rows out of 8,736: a cheap price for a result you can defend.

In [ ]:
CUT = pd.Timestamp("2025-09-01")
EMBARGO = pd.Timedelta("24h")

train = F.loc[:CUT - EMBARGO]
test  = F.loc[CUT + EMBARGO:]

assert train.index.max() < test.index.min(), "leakage: training overlaps test"

print("train:", train.index.min(), "->", train.index.max(), f"({len(train)} rows)")
print("test :", test.index.min(),  "->", test.index.max(),  f"({len(test)} rows)")
print("embargoed rows:", len(F) - len(train) - len(test))

### The detector

Scaler and model bound into one pipeline, so the scaler fitted on training rows is the one applied at prediction time. `random_state` is set, or the results are not reproducible.

**What to read in the output.** The threshold and the flag count. With `contamination=0.005` on 8,736 rows, expect roughly 44 flagged hours. If you asked for 0.5% and got thousands of flags, something is wrong — that sanity check is one of the four verification steps from topic 1.

The ten most anomalous hours should cluster in the labelled event windows. Check the dates against the answer key only after you have looked at them yourself.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(
    StandardScaler(),
    IsolationForest(n_estimators=200, contamination=0.005, random_state=42),
)
model.fit(train)

# continuous score across the whole year - lower means more anomalous
score = pd.Series(model.decision_function(F), index=F.index)

# threshold chosen from the false-alarm budget, as in week 4
thresh = score.quantile(0.005)
flags = score < thresh

print(f"threshold      : {thresh:.4f}")
print(f"flagged hours  : {int(flags.sum())} of {len(F)}")
print("\nten most anomalous hours:")
print(score.nsmallest(10).round(4))

### Explaining a flag

A score with no reason attached will not survive contact with a duty engineer. Per-channel z-scores against the training baseline are the cheapest explanation available.

**What to read in the output.** For each of the eight most anomalous hours, how far every channel sat from its training-period normal. The largest one or two z-scores per row are your explanation — the sentence you give the duty engineer instead of "the algorithm says so".

Expect the EV02 and EV05 hours to show a large `battery_soc_pct` deviation with array current near normal, which is exactly the relationship break described in the labels.

In [ ]:
raw_cols = ["solar_array_current_a", "bus_voltage_v", "battery_soc_pct",
            "battery_temp_c", "twta_temp_c", "rf_drive_pct"]

base = bus.loc[train.index, raw_cols]
z = (bus[raw_cols] - base.mean()) / base.std()

worst = score.nsmallest(8).index
print("per-channel z-scores for the eight most anomalous hours:")
z.loc[worst].round(1)

---

## Scoring, and the comparison the deliverable asks for

Event-level scoring, unchanged from Week 4: one flag anywhere inside a labelled window counts as catching that event, once. Anything outside every labelled window is a false alarm.

State these conventions in your write-up before computing anything:

| Decision | Convention |
|---|---|
| Partial catch of a multi-hour event | Counts as one catch |
| Repeated flags inside one event | Collapse to a single detection |
| Tolerance | ±1 hour either side of the labelled window |
| Leading metric | Recall where a miss risks hardware, precision where alarm fatigue is the risk |
| Why not accuracy | 5 events in 8,736 hours. A detector that flags nothing scores 99.94% |

**What this cell does.** Defines the event-level scorer. For each labelled event it widens the window by the tolerance and asks whether the detector fired anywhere inside; one hit counts as catching that event, once. Every flag outside all labelled windows is a false alarm. Returns a per-event detail table plus the summary metrics.

**What to read in the output.** The detail table shows which events were caught and whether each was univariately detectable. A good engineered detector should catch all five — including the three marked `no`.

In [ ]:
def score_events(flags, labels, tol="1H"):
    """Event-level precision and recall. flags: boolean Series, datetime index."""
    tol = pd.Timedelta(tol)
    caught, detail, seen = 0, [], pd.Series(False, index=flags.index)

    for _, row in labels.iterrows():
        w = ((flags.index >= row["start"] - tol) &
             (flags.index <= row["end"]   + tol))
        hit = bool(flags[w].any())
        caught += hit
        seen |= w
        detail.append({"event": row["event_id"], "type": row["type"],
                       "univariate": row["detectable_univariately"], "caught": hit})

    fa = int((flags & ~seen).sum())
    total = caught + fa
    return pd.DataFrame(detail), {
        "events": len(labels), "caught": caught, "false_alarms": fa,
        "recall":    round(caught / len(labels), 3),
        "precision": round(caught / total, 3) if total else float("nan"),
    }


detail, summary = score_events(flags, labels)
print(summary)
detail

### The honest comparison

Score the Week 4 statistical baseline on the **same** period with the **same** scorer. Without that, "the ML model is better" is an assertion rather than a finding.

The expected shape of the result: per-channel control charts catch EV01 and EV03 and nothing else; the engineered multivariate detector catches all five. A model that merely ties the baseline has not earned deployment — and recommending against your own model is a legitimate outcome.

**What to read in the output.** The comparison the weekly deliverable asks for: Week 4's per-channel control chart against Week 5's engineered multivariate detector, both fitted on the same training period and scored with the same function.

Expect the control chart to catch EV01 and EV03 only — recall around 0.4 — while the multivariate detector reaches all five. That gap is the case for deployment. If your numbers come out closer than this, the honest recommendation may be to keep the simpler detector.

In [ ]:
# week 4 baseline: per-channel 3-sigma control chart, fitted on the same training period
base_stats = bus.loc[train.index, raw_cols].agg(["mean", "std"])

dev = (bus[raw_cols] - base_stats.loc["mean"]).abs() / base_stats.loc["std"]
spc_flags = (dev > 3).any(axis=1).reindex(F.index).fillna(False)

_, spc_summary = score_events(spc_flags, labels)
_, ml_summary  = score_events(flags, labels)

pd.DataFrame([
    {"detector": "Control chart, per channel (week 4)", **spc_summary},
    {"detector": "Isolation forest, engineered (week 5)", **ml_summary},
]).set_index("detector")

---

## Change log

| Date | Change |
|---|---|
| 2026-09-15 | First issue for Week 5. Added `bus_telemetry_multivariate_hourly.csv` with six coupled channels and five embedded faults, three of them univariately undetectable, plus `anomaly_labels_week5.csv`. |

*Applied Machine Learning for Operations, Model Builder Track — prepared for NIGCOMSAT by Publica Academy.*